# ⚖️ Debate: Adversarial Multi-Agent Reasoning

Two agents argue opposite sides of a genuinely debatable question over several rounds, each rebutting the other,
and a neutral **Judge** synthesizes the result into a structured verdict.

## Learning Objectives
In this notebook, you will learn:
1. **Opposing personas** - Assign fixed stances with honest-argument rules
2. **Round-robin turns** - Alternate two debater nodes, with a round counter that decides when the debate ends
3. **Structured judging** - A Pydantic verdict with the strongest point from each side
4. **Running the debate** - Wire the graph, run it, and read the transcript and the verdict
5. **Cost vs value** - Compare a `2N + 1`-call debate with a single-call answer

## Where this fits
The blackboard notebooks (01–02) had agents *cooperate* on one artifact. Here they *disagree* on purpose, so the
judge sees the strongest case for both sides. `04_storm.ipynb` uses multiple perspectives for research instead of
argument.

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Liang et al., *Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate* ("MAD", EMNLP 2024). [arXiv:2305.19118](https://arxiv.org/abs/2305.19118) |
| Variant | Du et al., *Improving Factuality and Reasoning in Language Models through Multiagent Debate* (2023). [arXiv:2305.14325](https://arxiv.org/abs/2305.14325) |

### Definition

**Debate** is a multi-agent pattern in which two or more agents are assigned *opposing stances* on a genuinely debatable question and argue their side across several rounds, each rebutting the other's most recent point after seeing the full transcript so far. A neutral **Judge** agent then reviews the entire exchange and synthesizes a final answer that weighs the strongest points from each side.

The pattern follows the *debate* example in FareedKhan-dev's `all-agentic-architectures` collection: instead of asking a single LLM call for "the answer", you force the model to argue *both* sides of a hard question, then let a third, unaligned agent adjudicate. The adversarial pressure surfaces considerations a single pass would likely omit or under-weight.

**Which debate is this?** Two research setups share the name. This notebook follows Liang et al.'s *MAD*: the
debaters hold fixed affirmative and negative sides, and a judge reads the debate and extracts the final answer.
(MAD's judge can also end the debate early once it sees a correct answer; here the debate always runs
`MAX_ROUNDS`.) In Du et al.'s *multiagent debate*, every agent gets the same prompt and first answers on its own,
then reads the other agents' answers and revises its own over several rounds until they converge: no assigned
stances and no judge.

### High-Level Workflow

```
                 ┌────────────────────┐
        START ──▶│  Debater A (Pro)    │──┐
                 └────────────────────┘  │
                           ▲              │  full transcript so far
                           │              ▼
                 ┌────────────────────┐
                 │  Debater B (Con)    │
                 └────────────────────┘
                           │
                 (repeat for N rounds)
                           │
                           ▼
                 ┌────────────────────┐
                 │   Judge Agent        │──▶ END (structured verdict)
                 └────────────────────┘
```

Each debater node reads the *entire* transcript accumulated so far (not just the opponent's last turn) so it can maintain consistency with its own earlier arguments while directly rebutting the newest opposing point. A round counter drives a conditional edge that loops the two debaters back and forth until `max_rounds` is reached, then routes to the judge.

### When to Use

- Questions with **legitimate trade-offs** and no single objectively correct answer (architecture decisions, policy questions, build-vs-buy, strategy choices).
- When a single-agent answer tends to anchor on the first framing it generates and under-explore the counter-case.
- When you want an auditable trail showing *why* a recommendation was made, including the strongest counter-arguments that were considered and rejected.

### Strengths

- Forces explicit consideration of counter-arguments instead of one-sided reasoning.
- The judge's synthesis can be more nuanced and better-hedged than either debater's own conclusion.
- Transparent: the full transcript is a readable audit trail of the reasoning process.

### Weaknesses

- More expensive: `2N + 1` LLM calls instead of 1.
- Debaters can still both miss a consideration neither persona was primed to raise.
- Quality depends heavily on persona/system-prompt design — a weak "Con" persona produces a lopsided, unhelpful debate.
- The judge itself is a single point of failure/bias unless it is deliberately prompted to be even-handed.

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from typing import List, Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by both debaters and the judge
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🎭 Part 1: The Question and Two Opposing Personas

The question is a real technical trade-off with no single correct answer: *should a 6-person team building a new
B2B SaaS product start with microservices or a monolith?* Two debaters get fixed, opposite stances:
- **Pro-Microservices Advocate**: argues for microservices
- **Pro-Monolith Advocate**: argues for a monolith

Both must argue honestly (no strawmanning) and, after their opening turn, rebut the opponent's most recent point.
`MAX_ROUNDS` sets how many rounds of two turns run before the judge, so the debate costs `2 × MAX_ROUNDS + 1` LLM
calls. Part 5 then asks the same question in a single call.

In [ ]:
# ============================================================================
# DEBATE QUESTION & PERSONAS
# ============================================================================
DEBATE_QUESTION = (
    "Should a 6-person engineering team building a new B2B SaaS product "
    "adopt a microservices architecture or a monolith for their first version?"
)

PRO_MICROSERVICES_PERSONA = """You are the Pro-Microservices Advocate in a structured debate.
You must argue IN FAVOR of adopting a microservices architecture for the question at hand.

Rules:
- Argue persuasively but honestly. Do not invent facts or strawman the opposing side.
- If this is not your opening turn, you MUST directly rebut the single strongest point your
  opponent made in their most recent turn before advancing your own argument.
- Keep your turn focused: one clear rebuttal (if applicable) + one or two new supporting points.
- Keep responses to at most 120 words.
- Never break character or acknowledge you are an AI."""

PRO_MONOLITH_PERSONA = """You are the Pro-Monolith Advocate in a structured debate.
You must argue IN FAVOR of adopting a monolithic architecture for the question at hand.

Rules:
- Argue persuasively but honestly. Do not invent facts or strawman the opposing side.
- If this is not your opening turn, you MUST directly rebut the single strongest point your
  opponent made in their most recent turn before advancing your own argument.
- Keep your turn focused: one clear rebuttal (if applicable) + one or two new supporting points.
- Keep responses to at most 120 words.
- Never break character or acknowledge you are an AI."""

MAX_ROUNDS = 3

print(f"✅ Two personas, {MAX_ROUNDS} rounds -> {2 * MAX_ROUNDS} debater turns + 1 judge call = {2 * MAX_ROUNDS + 1} LLM calls")

---
## 🔁 Part 2: Round-Robin Turns

The state holds the transcript and a round counter. `render_transcript()` turns the transcript into prompt text, and
each debater node reads the **whole** transcript, so it can stay consistent with its own earlier turns while it
rebuts the newest point. After the Pro-Monolith turn, an `advance_round` node adds 1 to the counter, and
`should_continue_debate` sends the graph back to Pro-Microservices or on to the judge. That routing function is plain
Python, so the last cell of this part checks it without an LLM call.

In [ ]:
# ============================================================================
# STATE DEFINITION
# ============================================================================
class Turn(TypedDict):
    round: int
    speaker: str
    content: str


class DebateState(TypedDict):
    question: str
    transcript: List[Turn]
    round_number: int
    max_rounds: int
    verdict: dict

print("✅ DebateState keys:", list(DebateState.__annotations__))

In [ ]:
# ============================================================================
# HELPER: RENDER TRANSCRIPT FOR A PROMPT
# ============================================================================
def render_transcript(transcript: List[Turn]) -> str:
    if not transcript:
        return "(No arguments have been made yet. You are opening the debate.)"
    lines = [f"[Round {t['round']}] {t['speaker']}: {t['content']}" for t in transcript]
    return "\n\n".join(lines)

# The opening debater sees this placeholder instead of an empty transcript
print(render_transcript([]))

In [ ]:
# ============================================================================
# DEBATER NODES
# ============================================================================
def make_debater_node(persona_prompt: str, speaker_name: str):
    """Factory that builds a debater node bound to a fixed persona/stance."""

    def debater_node(state: DebateState) -> dict:
        transcript_so_far = render_transcript(state["transcript"])
        human_prompt = (
            f"Debate question: {state['question']}\n\n"
            f"Transcript so far:\n{transcript_so_far}\n\n"
            "Give your next turn now."
        )
        response = llm.invoke(
            [SystemMessage(content=persona_prompt), HumanMessage(content=human_prompt)]
        )
        new_turn: Turn = {
            "round": state["round_number"],
            "speaker": speaker_name,
            "content": response.content,
        }
        return {"transcript": state["transcript"] + [new_turn]}

    return debater_node


pro_microservices_node = make_debater_node(
    PRO_MICROSERVICES_PERSONA, "Pro-Microservices Advocate"
)
pro_monolith_node = make_debater_node(PRO_MONOLITH_PERSONA, "Pro-Monolith Advocate")


def advance_round(state: DebateState) -> dict:
    # A "round" completes once both debaters have spoken; increment after Monolith's turn.
    return {"round_number": state["round_number"] + 1}


def should_continue_debate(state: DebateState) -> Literal["pro_microservices", "judge"]:
    if state["round_number"] > state["max_rounds"]:
        return "judge"
    return "pro_microservices"

# Check the round logic without an LLM call: after round r, advance_round has set round_number to r + 1
routes = [should_continue_debate({"round_number": r + 1, "max_rounds": MAX_ROUNDS}) for r in range(1, MAX_ROUNDS + 1)]
for r, route in enumerate(routes, start=1):
    print(f"after round {r} -> {route}")
print(f"{'✅' if routes == ['pro_microservices'] * (MAX_ROUNDS - 1) + ['judge'] else '❌'} {MAX_ROUNDS} rounds, then the judge")

---
## ⚖️ Part 3: The Judge

The judge reads the full transcript and fills a Pydantic `DebateVerdict`: the strongest point from each side, the
weak or overstated claims, a recommendation and a confidence level. Its prompt tells it to reason past the debate
rather than pick a winner, so the recommendation may be more conditional than either side's position.

In [ ]:
# ============================================================================
# JUDGE AGENT (STRUCTURED OUTPUT)
# ============================================================================
class DebateVerdict(BaseModel):
    """Structured verdict produced by the neutral Judge after reviewing a full debate transcript."""

    strongest_pro_microservices_point: str = Field(
        description="The single strongest, most defensible point raised by the Pro-Microservices Advocate."
    )
    strongest_pro_monolith_point: str = Field(
        description="The single strongest, most defensible point raised by the Pro-Monolith Advocate."
    )
    weaknesses_identified: str = Field(
        description="Brief note on any weak, unsupported, or overstated claims made by either side."
    )
    final_recommendation: str = Field(
        description=(
            "A final, nuanced, synthesized recommendation that may be more conditional/balanced "
            "than either single debater's position (e.g. 'it depends on X, Y' or a staged approach)."
        )
    )
    confidence: str = Field(
        description="Judge's confidence in the recommendation: 'low', 'medium', or 'high'."
    )


judge_llm = llm.with_structured_output(DebateVerdict)

JUDGE_SYSTEM_PROMPT = """You are a neutral, impartial Judge reviewing a structured debate.
You have no stake in either side's position. Your job is to:
1. Identify the single strongest point each side made.
2. Note any weak or overstated claims from either side.
3. Produce a final recommendation that is allowed to be MORE nuanced than either debater's
   position -- for example, a conditional answer, a staged/phased approach, or a synthesis
   that borrows the best ideas from both sides.
Do not simply declare one side the 'winner' -- your job is to reason past the debate, not referee it."""


def judge_node(state: DebateState) -> dict:
    transcript_so_far = render_transcript(state["transcript"])
    human_prompt = (
        f"Debate question: {state['question']}\n\n"
        f"Full transcript:\n{transcript_so_far}\n\n"
        "Produce your structured verdict now."
    )
    verdict: DebateVerdict = judge_llm.invoke(
        [SystemMessage(content=JUDGE_SYSTEM_PROMPT), HumanMessage(content=human_prompt)]
    )
    return {"verdict": verdict.model_dump()}

print("✅ Judge returns:", list(DebateVerdict.model_fields))

---
## ▶️ Part 4: Running the Debate

The graph is `START → pro_microservices → pro_monolith → advance_round`; from there the conditional edge loops back
to `pro_microservices` or moves on to `judge → END`. The printed edge list shows that round-robin shape. The run then
fills the transcript, which the next two cells print with the judge's verdict.

In [ ]:
# ============================================================================
# BUILD THE DEBATE GRAPH
# ============================================================================
debate_graph = StateGraph(DebateState)

debate_graph.add_node("pro_microservices", pro_microservices_node)
debate_graph.add_node("pro_monolith", pro_monolith_node)
debate_graph.add_node("advance_round", advance_round)
debate_graph.add_node("judge", judge_node)

debate_graph.add_edge(START, "pro_microservices")
debate_graph.add_edge("pro_microservices", "pro_monolith")
debate_graph.add_edge("pro_monolith", "advance_round")
debate_graph.add_conditional_edges(
    "advance_round",
    should_continue_debate,
    {"pro_microservices": "pro_microservices", "judge": "judge"},
)
debate_graph.add_edge("judge", END)

debate_app = debate_graph.compile()

print("✅ Debate graph compiled")

In [ ]:
# ============================================================================
# INSPECT THE GRAPH
# ============================================================================
for edge in debate_app.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

In [ ]:
# ============================================================================
# RUN THE DEBATE
# ============================================================================
initial_state: DebateState = {
    "question": DEBATE_QUESTION,
    "transcript": [],
    "round_number": 1,
    "max_rounds": MAX_ROUNDS,
    "verdict": {},
}

result = debate_app.invoke(initial_state)

print(f"✅ Debate finished: {len(result['transcript'])} turns over {MAX_ROUNDS} rounds; verdict fields: {list(result['verdict'])}")

In [ ]:
# ============================================================================
# PRINT THE FULL TRANSCRIPT
# ============================================================================
print(f"DEBATE QUESTION:\n{result['question']}\n")
print("=" * 80)
for turn in result["transcript"]:
    print(f"\n--- Round {turn['round']} | {turn['speaker']} ---")
    print(turn["content"])
print("\n" + "=" * 80)

In [ ]:
# ============================================================================
# PRINT THE JUDGE'S VERDICT
# ============================================================================
verdict = result["verdict"]
print("JUDGE'S VERDICT\n" + "-" * 80)
print(f"Strongest Pro-Microservices point:\n  {verdict['strongest_pro_microservices_point']}\n")
print(f"Strongest Pro-Monolith point:\n  {verdict['strongest_pro_monolith_point']}\n")
print(f"Weaknesses identified:\n  {verdict['weaknesses_identified']}\n")
print(f"Final synthesized recommendation:\n  {verdict['final_recommendation']}\n")
print(f"Judge confidence: {verdict['confidence']}")

**Reading the output:** after the opening turn, each turn should start by answering the opponent's last point. If
both sides only restate their own case, the persona rules need tightening. The judge's `final_recommendation` may be
conditional ("start with a monolith, split out services when …"). That is the point of a judge that reasons past the
debate instead of naming a winner.

---
## 🆚 Part 5: Cost vs Value — the Same Question in One Call

The debate's value only shows against something. Here is the same question answered by a single LLM call with a
neutral prompt, followed by the call count of each approach.

In [ ]:
# ============================================================================
# SINGLE-CALL BASELINE
# ============================================================================
single_answer = llm.invoke(
    f"{DEBATE_QUESTION}\n\nGive a recommendation in at most 150 words."
).content
print("SINGLE-CALL ANSWER\n" + "-" * 80)
print(single_answer)

print("\nDEBATE JUDGE'S RECOMMENDATION\n" + "-" * 80)
print(result["verdict"]["final_recommendation"])

debate_calls = len(result["transcript"]) + 1
print(f"\nLLM calls - debate: {debate_calls} ({len(result['transcript'])} turns + 1 judge), single answer: 1")

### Reading the output

Compare the two recommendations printed above:

1. **Each debater's own turns** argue one side only, because that is the only side each one was assigned.
2. **The single-call answer** may already be balanced. A capable model often hedges on its own, so the debate
   does not always "win".
3. **The judge's `final_recommendation`** has seen both sides fully argued and rebutted. Look for what it adds:
   named conditions (team size, scaling pressure), a staged path, or a counter-argument the single answer never
   mentioned.

The value of the pattern is the **explicit, auditable trail** (the transcript plus the strongest point from each
side). The cost is shown in the last line: `2N + 1` calls instead of one. For a factual question with one correct
answer, the single call is the better choice.

---
## 📝 Summary

- **Debate** assigns opposing personas/stances to two or more agents and lets them argue over multiple
  rounds, each seeing the full transcript so far and directly rebutting the opponent's latest point.
- A **neutral Judge agent**, prompted specifically to synthesize rather than simply pick a "winner",
  produces a structured verdict (via Pydantic / `with_structured_output`) that can be more conditional than
  either debater's position.
- This is **MAD-style** debate (Liang et al.): fixed sides plus a judge. Du et al.'s multiagent debate drops both:
  agents start from their own answers and revise them until they agree.
- The control flow is a small **round-robin `StateGraph`**: two debater nodes alternate, a round counter
  drives a conditional edge, and once `max_rounds` is reached the graph routes to the judge node and ends.
- The pattern trades extra LLM calls (`2N + 1` for `N` rounds) for a more defensible, auditable answer on
  genuinely debatable questions — it is not worth the cost on simple factual questions with one correct answer.
- Persona/system-prompt design matters a lot: both debaters must be instructed to argue honestly and to
  engage with (not ignore) the opponent's strongest point, or the debate degenerates into two parallel
  monologues with nothing for the judge to actually adjudicate.

### Next Steps
- `04_storm.ipynb`: several perspectives cooperate to *research and write* instead of argue